# Testing scipy linear programming

## Imports, definitions and base tests

In [ ]:
# Module imports
import sys

import numpy as np
import scipy as sp
from loguru import logger



In [ ]:
# Set up the logger
logger.remove()
logger.add(
    sink=sys.stdout,
    format="<level>{level:<10} | {message}</>",
    level="INFO",
    colorize=True,
)

### Behaviors as vectors and notation

Suppose a (2,2,2) routed Bell experiment.

$ p(ab|xyz) \in \mathbb{R}^{32}$ is the observed behavior, assumed to be no-signaling : $$p \in \mathcal{NS}$$

We wish for easy conversion from a vector format (for algebraic operations) to a matrix format (for leggibility) :
$$p=\begin{pmatrix} p_{00|00S} \\ p_{00|01S} \\ \vdots \\ p_{00|00L} \\ \vdots \end{pmatrix}\quad \leftrightarrow \quad p=\begin{pmatrix} p_{00|00}& p_{00|01}& p_{00|10}& p_{00|11}\\ p_{01|00}& p_{01|01}& \dots\\ \vdots & & \ddots \\ & & & p_{11|11} \end{pmatrix}$$
where, to account for both matrices $p(z=S)$ and $p(z=L)$, both cases are represented in two matrices, making $p$ either a column vector in $\mathbb{R}^{32}$ or a third-order tensor of shape $(2,4,4)$.

We defined in the Behavior class (behavior.py) some utility functions.

In [ ]:
from behaviors import Behavior

# The maximally mixed state over the experiment space
I = Behavior((1 / 4) * np.ones(32))  # noqa: E741

# The usual (2,2,2) PR box
SR_pr_box = np.array(
    [ 1/2, 1/2, 1/2, 0, 0, 0, 0, 1/2, 0, 0, 0, 1/2, 1/2, 1/2, 1/2, 0]
)

# The PR box in the experiment space : p(ab|xy) is assumed to be
# independent of the value of z
pr_box = Behavior(np.concatenate((SR_pr_box, SR_pr_box), axis=0))

def to_behavior(arr):
    return Behavior(np.concatenate((arr, arr), axis=0))

non_positive_arr = np.array(
    [-1/2,-1/2,-1/2,0,0,0,0,1/2,0,0,0,1/2,1/2,1/2,1/2,0,],
)
non_normalized_arr = np.array(
    [1/2,1/2,1/2,0,0,0,0,0,0,0,0,1/2,1/2,1/2,1/2,0,]
)
non_ns_arr = np.array(
    [1/2,1/2,0,0,0,0,1/2,1/2,1/2,1/2,0,0,0,0,1/2,1/2,]
)

non_positive = to_behavior(non_positive_arr)
non_normalized = to_behavior(non_normalized_arr)
non_ns = to_behavior(non_ns_arr)


In [ ]:
# Sanity check cell, don't mind me

# print("PR box: ", pr_box)
# print("I: ", I)

print("I == I: ", I == I)
print("I == pr_box: ", I == pr_box)
print("I == 0: ", I == 0)
print("I == 0.25: ", I == 0.25)
print("pr_box == pr_box_array: ", pr_box == np.concatenate((SR_pr_box, SR_pr_box), axis=0))

print("I positive: ", I.positivity())
print("I normalized: ", I.normalization())
print("I no-signaling: ", I.no_signaling())

print("PR box positive: ", pr_box.positivity())
print("PR box normalized: ", pr_box.normalization())
print("PR box no-signaling: ", pr_box.no_signaling())

print("Non-positive: ", non_positive.positivity())
print("Non-normalized: ", non_normalized.normalization())
print("Non-no-signaling: ", non_ns.no_signaling())

print("Non-no-signaling is normalized: ", non_ns.is_normalized())

In [ ]:
# Checking the indices correspondence functions

from behaviors import routed_index_to_indices, routed_indices_to_index

for i in range(2*2**2*2**2):
    print(f"i={i} -> routed index: {routed_index_to_indices(i, m=2)}, computed i: {routed_indices_to_index(*routed_index_to_indices(i,m=2), m=2)}")  # noqa: E501

In [ ]:
# Checking that the no-signaling set is well defined in no_signaling_set.py

from no_signaling_set import routed_no_signaling_equations
from behaviors import completely_mixed_behavior, pr_box

delta, m = 2, 2
equations, right_side = routed_no_signaling_equations(delta, m)


# print(f"Delta: {delta}, m: {m}")
# print("\n--------------\n")
# print(f"Equations shape: {equations.shape}")
# print("\n--------------\n")

# for line in equations:
#     print(str(line).strip("[]").replace("\n", "").replace(" ", "").replace("-1", "2").replace(".", "").replace("0", "."))  # noqa: E501

# print("\n--------------\n")
# print(f"Right side shape: {right_side.shape}")
# print("\n--------------\n")
# print(f"Right side: {right_side}")


print(np.all(equations @ completely_mixed_behavior.get_vector() == right_side))
print(np.all(equations @ pr_box.get_vector() == right_side))



## Sampling behaviors

### Uniform on normalized behaviors $\mathcal{B}$

In [ ]:
from samplers import UniformNormalizedSampler

sampler = UniformNormalizedSampler(2,2,True) # Samples normalized behaviors

sample_behavior = sampler.sample()
print("Sampled behavior: ", sample_behavior)
print("Sample behavior is normalized: ", sample_behavior.is_normalized())
print("Sample behavior is no-signaling: ", sample_behavior.no_signaling())

Sampling no-signaling behaviors can't reasonably be achieved with rejection sampling from normalized behaviors though, since as $dim(\mathcal{NS}) < dim(\mathcal{B})$, the usual measure of $\mathcal{NS}$ in $\mathcal{B}$ is null. Getting a no-signaling behavior would thus be very, very lucky.

A consequence of this is that we will need to implement uniform sampling on the $\mathcal{NS}$ polytope to sample no-signaling behaviors directly. This can be achieved if we know the polytope's vertices, which is the case in low-dimensions only. The method consists in partitioning the arbitrary bounded polytope in simplices, which we can sample from by weighing them using their volumes, and then using simplex-specific methods to sample uniformly in the chosen simplex.

It also seems that we are able to uniformly sample from a polytope using MCMC methods, as per [Sun and Chen, 2024](https://arxiv.org/abs/2412.06629), using the ``polytopewalk`` module.

### Testing ``polytopewalk``

In [ ]:
import polytopewalk as pw

help(pw.sparse.SparseRandomWalk.generateCompleteWalk)

In [ ]:
from samplers import NoSignalingSampler

ns_sampler = NoSignalingSampler(2,2,True) # Samples no-signaling behaviors
samples_og = ns_sampler.sample_multiple()

In [ ]:
from samplers import SamplesAnalyzer

analyzer = SamplesAnalyzer(samples_og)

### Plotting the uniform sampling in a random plane

In [ ]:
analyzer.plot_projection()

### Checking uniformity

In [ ]:
# # Apply it to the samples
# stats = analyzer.global_analyze_sampling(plot=True)
# Apply it to the samples
local_stats = analyzer.analyze_local_uniformity(k=20, plot=True)
